# M3L4 E06 — Evaluator Agent simple [OK] Resolution
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

## ¿Por qué importa este ejercicio?

Hasta ahora medimos **precisión del routing** (qué agente se elige). Pero un agente puede rutear correctamente y aún así dar una **mala respuesta**: que no cubra los puntos clave, que alucine, o que sea genérica.

Necesitamos un **Evaluator Agent**: un sistema que recibe la respuesta del agente y la califica según qué tan bien cubre los temas esperados.

| Concepto | Definición simple | Cómo aparece acá |
|---|---|---|
| **Evaluator** | Sistema que juzga la calidad de una respuesta | `simple_keyword_evaluator()` |
| **Keywords esperadas** | Términos que DEBEN aparecer en la respuesta | Lista de palabras clave por caso |
| **Score** | Proporción de keywords encontradas | `matched / total_expected` |
| **LLM-as-judge** | Evaluador usando un modelo de lenguaje | No implementado (sería el paso siguiente) |

In [ ]:
import pandas as pd

def simple_keyword_evaluator(expected_keywords: list, actual_answer: str) -> dict:
    if not expected_keywords:
        return {'score': 0, 'matched_keywords': [], 'reason': 'No expected keywords provided.'}

    matched = [kw for kw in expected_keywords if kw.lower() in actual_answer.lower()]
    score = round(len(matched) / len(expected_keywords), 2)

    return {
        'score': score,
        'matched_keywords': matched,
        'reason': f'Matched {len(matched)} of {len(expected_keywords)} expected keywords.'
    }

print('Función lista.')

## Solución — Pruebas del evaluador

Probamos 4 escenarios:
1. **Perfecto**: todas las keywords están en la respuesta -> score = 1.0
2. **Parcial**: algunas keywords aparecen, otras no -> score parcial
3. **Incorrecto**: ninguna keyword aparece -> score = 0.0
4. **Sin keywords**: lista vacía -> score = 0.0 (caso borde)

In [ ]:
result1 = simple_keyword_evaluator(['factura', 'pagos', 'portal'], 'Podés ver tu factura desde el portal de pagos.')
print('Caso 1 (perfecto):', result1)

result2 = simple_keyword_evaluator(['vacaciones', 'solicitud', 'formulario', 'portal'], 'Para pedir vacaciones completá el formulario.')
print('Caso 2 (parcial):', result2)

result3 = simple_keyword_evaluator(['factura', 'pagos', 'portal'], 'Probá reiniciar la app.')
print('Caso 3 (incorrecto):', result3)

result4 = simple_keyword_evaluator([], 'Probá reiniciar la app.')
print('Caso 4 (sin keywords):', result4)

## Evaluación de múltiples casos

Probamos el evaluador contra 5 casos reales del sistema multiagente.

In [ ]:
evaluation_cases = [
    {'query': 'Necesito ver mi factura del mes pasado',   'expected_keywords': ['factura', 'portal', 'pagos'],      'agent_response': 'Podés ver tu factura desde el portal de pagos.'},
    {'query': '¿Cómo solicito mis días de vacaciones?',   'expected_keywords': ['vacaciones', 'portal', 'formulario'], 'agent_response': 'Para solicitar vacaciones ingresá al portal de RRHH y completá el formulario.'},
    {'query': 'Mi VPN no conecta desde ayer',             'expected_keywords': ['VPN', 'conexión', 'soporte'],       'agent_response': 'Probá reiniciar el router.'},
    {'query': '¿Cuándo se procesa el reembolso de gastos?', 'expected_keywords': ['reembolso', 'gastos', '48 horas'], 'agent_response': 'Los gastos se procesan dentro de las 48 horas hábiles.'},
    {'query': 'Necesito el contrato de confidencialidad', 'expected_keywords': ['contrato', 'confidencialidad', 'legal'], 'agent_response': 'Completá el formulario de RRHH para acceder al contrato.'},
]

results = []
for case in evaluation_cases:
    er = simple_keyword_evaluator(case['expected_keywords'], case['agent_response'])
    results.append({'query': case['query'][:40], 'score': er['score'], 'matched': er['matched_keywords'], 'reason': er['reason']})

df = pd.DataFrame(results)
print(f'Quality score promedio: {df["score"].mean():.2f}')
df

## Verificación

In [ ]:
r = simple_keyword_evaluator(['factura', 'pagos', 'portal'], 'Podés ver tu factura desde el portal de pagos.')
assert r['score'] == 1.0
assert len(r['matched_keywords']) == 3
r2 = simple_keyword_evaluator([], 'cualquier cosa')
assert r2['score'] == 0
print('Checks E06 OK')

## [OK] Cierre — ¿Qué logramos?

| Método | Ventaja | Limitación |
|---|---|---|
| **Keyword evaluator** | Simple, rápido, determinístico | No entiende sinónimos ni contexto |
| **LLM-as-judge** (próximo paso) | Entiende semántica, sinónimos, tono | Más lento, más caro, no determinístico |

> **Cuándo usar cada uno:** el keyword evaluator es ideal para validación rápida en desarrollo. Para producción, combinalo con LLM-as-judge para casos ambiguos.

**¿Qué sigue?** En E07 vamos a armar un **dashboard local de métricas** para visualizar accuracy, latencia y calidad en un solo lugar.